# AI4SAW — Notebook 03: RAG Q&A and Silence Detection Demo

Demonstrates:
1. **RAG Q&A** — MMR retrieval → cross-encoder re-ranking → cited answer generation
2. **Silence Detection Approach A** — expectation gap (CDISaW/ACLED events vs retrieval confidence)
3. **Silence Detection Approach B** — density mapping (corpus density vs conflict intensity)
4. **Dataset export** — all structured outputs to JSON/GeoJSON

In [ ]:
import sys
sys.path.insert(0, '..')

from ai4saw.core.config import settings
print(f'Provider: {settings.provider} | Model: {settings.default_model}')

## 1. RAG Q&A with source citations

In [ ]:
from ai4saw.retrieval.qa import answer

question = 'What forms of forced labour were documented in Bosnian detention camps?'

print(f'Question: {question}\n')
response = answer(question)

print('=== ANSWER ===')
print(response.answer)
print()
print(f'Retrieved: {response.retrieved_chunks} chunks → re-ranked to {response.reranked_to}')
print(f'Confidence: {response.confidence:.2f}')
print()
print('=== SOURCES ===')
for i, src in enumerate(response.sources, 1):
    print(f'[Source {i}] {src.source_filename} | {src.geography} | {src.date_published}')

## 2. Benchmark Q&A questions

In [ ]:
import json
from pathlib import Path

questions_path = Path('../eval/testdata/rag_questions.json')
questions = json.loads(questions_path.read_text())

print(f'Running {len(questions)} benchmark questions...\n')

results = []
for item in questions[:3]:   # limit to 3 for demo speed
    q = item['question']
    print(f'Q: {q}')
    resp = answer(q)
    print(f'A: {resp.answer[:200]}...')
    print(f'   conf={resp.confidence:.2f}  sources={len(resp.sources)}\n')
    results.append({'question': q, 'answer': resp.answer, 'confidence': resp.confidence})

## 3. Silence Detection — Approach A (Expectation Gap)

Supply a list of known CDISaW/ACLED events and measure retrieval confidence for each.
High conflict intensity + low retrieval confidence = informational silence candidate.

In [ ]:
from ai4saw.synthesis.silence import detect_silence_expectation_gap

# Example reference events — in practice, load from CDISaW/ACLED CSV
reference_events = [
    {
        'event_id': 'CDISaW-001',
        'location': 'Srebrenica',
        'date': '1995-07-11',
        'conflict_intensity': 0.95,
    },
    {
        'event_id': 'CDISaW-002',
        'location': 'Foča',
        'date': '1992-08-01',
        'conflict_intensity': 0.88,
    },
    {
        'event_id': 'CDISaW-003',
        'location': 'El Geneina',
        'date': '2023-04-15',
        'conflict_intensity': 0.91,
    },
    {
        'event_id': 'ACLED-1042',
        'location': 'Prijedor',
        'date': '1992-05-30',
        'conflict_intensity': 0.82,
    },
    {
        'event_id': 'ACLED-2091',
        'location': 'Nyala',
        'date': '2023-06-01',
        'conflict_intensity': 0.75,
    },
]

candidates = detect_silence_expectation_gap(reference_events)

print(f'Silence candidates (ranked by score):\n')
for c in candidates:
    bar = '█' * int(c.silence_score * 20)
    print(f'  {c.event_id:15s}  {c.location:15s}  silence={c.silence_score:+.3f}  {bar}')
    print(f'    intensity={c.conflict_intensity:.2f}  ret_conf={c.retrieval_confidence:.2f}')
    print(f'    reason: {c.candidate_reason}\n')

## 4. Silence Detection — Approach B (Density Mapping)

Cluster the corpus by geography and time window, compare density against ACLED intensity.

In [ ]:
from ai4saw.synthesis.silence import detect_silence_density_map

# Example ACLED reference intensities — in practice, load from ACLED export
# Keys are '{geography}_{year}-Q{quarter}'
acled_reference = {
    'Bosnia_1992-Q2': 0.90,
    'Bosnia_1992-Q3': 0.88,
    'Bosnia_1995-Q3': 0.95,
    'Sudan_2023-Q2': 0.91,
    'Sudan_2023-Q3': 0.87,
    'unknown_unknown': 0.0,
}

density_cells = detect_silence_density_map(acled_reference)

print(f'Density cells ({len(density_cells)} total):\n')
for cell in density_cells[:10]:
    bar = '█' * int(max(cell.silence_score, 0) * 20)
    print(
        f'  {cell.geography:15s}  {cell.time_window:10s}  '
        f'docs={cell.document_count:3d}  chunks={cell.chunk_count:4d}  '
        f'intensity={cell.conflict_intensity:.2f}  silence={cell.silence_score:+.3f}  {bar}'
    )

## 5. Export all structured outputs

In [ ]:
from ai4saw.synthesis.export import export_silences
from pathlib import Path

# Export silence candidates
silence_path = export_silences(candidates)
print(f'Silences exported to: {silence_path}')

# Read it back to confirm
import json
data = json.loads(silence_path.read_text())
print(f'\nTop silence candidate:')
print(json.dumps(data[0], indent=2))

## 6. RAGAS evaluation (requires indexed corpus)

Runs RAGAS faithfulness, answer relevance, context precision, and context recall.

In [ ]:
# Uncomment to run — requires ragas and a populated corpus
# from eval.rag_eval import run_ragas_eval
# report = run_ragas_eval(
#     questions_path=Path('../eval/testdata/rag_questions.json'),
#     output_path=Path('../eval/results/rag_eval.json'),
# )
# print(json.dumps(report['metrics'], indent=2))